# Notebook to analyse flood results

## Clip city limits

In [32]:
import arcpy
import pandas as pd

aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

# Scenario group layer
group_layer_name = "Flood_results_shape"
group_layer = next((lyr for lyr in m.listLayers() if lyr.isGroupLayer and lyr.name == group_layer_name), None)
if group_layer is None:
    raise Exception(f"Group layer '{group_layer_name}' not found.")

scenario_layers = [lyr for lyr in group_layer.listLayers() if lyr.isFeatureLayer]

# City boundary layer (from map)
boundary_layer_name = "Municipal_boundaries_Project"
boundary_lyr = next((lyr for lyr in m.listLayers() if lyr.isFeatureLayer and lyr.name == boundary_layer_name), None)
if boundary_lyr is None:
    raise Exception(f"Boundary layer '{boundary_layer_name}' not found in the map.")

print("Scenarios:", [l.name for l in scenario_layers])
print("Boundary layer:", boundary_lyr.name)


Scenarios: ['GDB_2yearflood_current', 'GDB_10yearflood_current', 'GDB_100yearflood_current', 'GDB_2yearflood_future', 'GDB_10yearflood_future', 'GDB_100yearflood_future']
Boundary layer: Municipal_boundaries_Project


In [33]:
# Where to store clipped outputs
out_gdb = aprx.defaultGeodatabase

results_city = []

for lyr in scenario_layers:
    out_fc = out_gdb + "\\" + f"{lyr.name}_CITY"

    # Clip polygons to city boundary
    arcpy.analysis.Clip(in_features=lyr, clip_features=boundary_lyr, out_feature_class=out_fc)

    # Sum area inside city
    total_area_ft2 = 0.0
    with arcpy.da.SearchCursor(out_fc, ["Shape_Area"]) as cursor:
        for (a,) in cursor:
            if a is not None:
                total_area_ft2 += float(a)

    results_city.append({
        "Scenario": lyr.name,
        "FloodExtentCity_ft2": total_area_ft2,
        "FloodExtentCity_acres": total_area_ft2 / 43560
    })

df_extent_city = pd.DataFrame(results_city).sort_values("Scenario").reset_index(drop=True)
df_extent_city


,Scenario,FloodExtentCity_ft2,FloodExtentCity_acres
0,GDB_100yearflood_current,3.260815e+07,748.580085
1,GDB_100yearflood_future,3.525054e+07,809.241100
2,GDB_10yearflood_current,1.732977e+07,397.836809
3,GDB_10yearflood_future,2.399425e+07,550.832191
4,GDB_2yearflood_current,8.112047e+06,186.226970
5,GDB_2yearflood_future,1.679160e+07,385.482015


## Compare current x future

In [34]:
# Separate current and future scenarios (CITY-LIMITED)
current = df_extent_city[~df_extent_city["Scenario"].str.contains("future")].copy()
future  = df_extent_city[df_extent_city["Scenario"].str.contains("future")].copy()

# Common key (return period)
current["RP"] = current["Scenario"].str.extract(r"(\d+year)")
future["RP"]  = future["Scenario"].str.extract(r"(\d+year)")

# Merge current and future
df_increase_city = current.merge(
    future,
    on="RP",
    suffixes=("_current", "_future")
)

# Calculate increase (acres)
df_increase_city["Increase_acres"] = (
    df_increase_city["FloodExtentCity_acres_future"]
    - df_increase_city["FloodExtentCity_acres_current"]
)

df_increase_city["Increase_percent"] = (
    df_increase_city["Increase_acres"]
    / df_increase_city["FloodExtentCity_acres_current"]
) * 100

# Keep only what matters for results
df_increase_city = df_increase_city[[
    "RP",
    "FloodExtentCity_acres_current",
    "FloodExtentCity_acres_future",
    "Increase_acres",
    "Increase_percent"
]]

df_increase_city


,RP,FloodExtentCity_acres_current,FloodExtentCity_acres_future,Increase_acres,Increase_percent
0,100year,748.580085,809.241100,60.661015,8.103477
1,10year,397.836809,550.832191,152.995382,38.456819
2,2year,186.226970,385.482015,199.255044,106.995804


In [35]:
print(current["Scenario"].tolist())
print(future["Scenario"].tolist())


['GDB_100yearflood_current', 'GDB_10yearflood_current', 'GDB_2yearflood_current']
['GDB_100yearflood_future', 'GDB_10yearflood_future', 'GDB_2yearflood_future']


## Compare to the city boundaries

In [36]:
# --- City area (acres) from Municipal_boundaries ---
city_area_ft2 = 0.0
with arcpy.da.SearchCursor(boundary_lyr, ["Shape_Area"]) as cursor:
    for (a,) in cursor:
        if a is not None:
            city_area_ft2 += float(a)

city_area_acres = city_area_ft2 / 43560
print("City area (acres):", city_area_acres)

# --- Add percent-of-city flooded to the city-limited flood extent table ---
df_city_share = df_extent_city.copy()
df_city_share["CityArea_acres"] = city_area_acres
df_city_share["PercentCityFlooded"] = (df_city_share["FloodExtentCity_acres"] / city_area_acres) * 100

# Results-ready view
df_city_share[["Scenario", "FloodExtentCity_acres", "PercentCityFlooded"]].sort_values("Scenario").reset_index(drop=True)


City area (acres): 2565.750418264262


,Scenario,FloodExtentCity_acres,PercentCityFlooded
0,GDB_100yearflood_current,748.580085,29.175873
1,GDB_100yearflood_future,809.241100,31.540133
2,GDB_10yearflood_current,397.836809,15.505671
3,GDB_10yearflood_future,550.832191,21.468658
4,GDB_2yearflood_current,186.226970,7.258187
5,GDB_2yearflood_future,385.482015,15.024143
